[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_00/MFM_ch0_cross_asset/MFM_ch0_cross_asset.ipynb)

# MFM_ch0_cross_asset

**Modelling Financial Markets (MFM) — Chapter 0: Financial Markets in 2026**

Growth of 1 USD invested in January 2015, all in USD: S&P 500 with dividends reinvested (SPY), Euro Stoxx 50 and Nikkei 225 price indices converted to USD with FRED exchange rates, gold, long US Treasuries (TLT, total return) and Bitcoin; the risk-return scatter (CAGR versus annualised volatility) for these assets plus Ethereum, EUR/USD and EUR/RON, 2015-2026; and the dividend gap between the S&P 500 total-return and price series.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_0'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Style and data loader

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def log_axis(ax, ticks):
    """Axa logaritmica cu etichete lizibile (fara notatie stiintifica)."""
    ax.set_yscale('log')
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())


In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
LOCAL_DIRS = ['../../../data/market', '../../data/market', '../data/market', 'data/market']   # repo clonat
START, END = '2000-01-01', '2026-09-18'

# nume -> (sursa, simbol, camp)
SERIES = {
    'S&P 500': ('market', 'GSPC.INDX', 'close'),
    'Euro Stoxx 50': ('market', 'STOXX50E.INDX', 'close'),
    'Nikkei 225': ('market', 'N225.INDX', 'close'),
    'VIX': ('market', 'VIX.INDX', 'close'),
    'BET-TR': ('market', 'BETTR.INDX', 'close'),
    'BET': ('market', 'BET', 'close'),
    'BET-XT': ('market', 'BETXT.INDX', 'close'),
    'US Treasuries 20y+ (TLT)': ('market', 'TLT.US', 'adjusted_close'),
    'SPY': ('market', 'SPY.US', 'adjusted_close'),
    'RSP': ('market', 'RSP.US', 'adjusted_close'),
    'IBIT': ('market', 'IBIT.US', 'close'),
    'IBIT volume': ('market', 'IBIT.US', 'volume'),
    'Gold': ('market', 'XAUUSD.FOREX', 'close'),
    'EUR/USD': ('market', 'EURUSD.FOREX', 'close'),
    'EUR/RON (market file)': ('market', 'EURRON.FOREX', 'close'),
    'Bitcoin': ('market', 'BTC-USD.CC', 'close'),
    'Ethereum': ('market', 'ETH-USD.CC', 'close'),
    'Romania 10y': ('market', 'RO10Y.GBOND', 'close'),
    'Germany 10y': ('market', 'DE10Y.GBOND', 'close'),
    'Banca Transilvania': ('market', 'TLV.RO', 'adjusted_close'),
    'OMV Petrom': ('market', 'SNP.RO', 'adjusted_close'),
    'BRD': ('market', 'BRD.RO', 'adjusted_close'),
    'Transgaz': ('market', 'TGN.RO', 'adjusted_close'),
    'Romgaz': ('market', 'SNG.RO', 'adjusted_close'),
    'Hidroelectrica': ('market', 'H2O.RO', 'adjusted_close'),
    'EUR/RON': ('bnr', 'EUR', None),
    'DGS10': ('fred', 'DGS10', None),
    'DGS2': ('fred', 'DGS2', None),
    'FEDFUNDS': ('fred', 'FEDFUNDS', None),
    'USD per EUR': ('fred', 'DEXUSEU', None),
    'JPY per USD': ('fred', 'DEXJPUS', None),
    'T10Y2Y': ('fred', 'T10Y2Y', None),
    'Stablecoins': ('defillama', 'stablecoincharts/all', None),
}


def read_market(symbol):
    """Fisierul de date de piata: local (repo clonat), altfel din repo-ul GitHub."""
    for d in LOCAL_DIRS:
        path = os.path.join(d, f'{symbol}.csv')
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True)
    return pd.read_csv(REPO_RAW + f'{symbol}.csv', index_col='date', parse_dates=True)


def fetch_daily(symbol, source='market', field='close', start=START, end=END):
    """Punctul unic de acces la date: intoarce o pd.Series zilnica (fara NaN)."""
    if source == 'market':
        s = read_market(symbol)[field]
    elif source == 'fred':
        raw = urllib.request.urlopen(
            f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={symbol}', timeout=90).read().decode()
        s = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True, na_values='.').iloc[:, 0]
    elif source == 'defillama':
        req = urllib.request.Request(f'https://stablecoins.llama.fi/{symbol}',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        d = json.load(urllib.request.urlopen(req, timeout=90))
        s = pd.Series({pd.to_datetime(int(x['date']), unit='s'): x['totalCirculatingUSD'].get('peggedUSD', 0)
                       for x in d}) / 1e9                                   # miliarde USD
    elif source == 'bnr':
        # cursul oficial BNR (RON pentru o unitate de valuta), arhive XML anuale
        import re
        rows = []
        for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
            url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=90).read().decode()
            for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
                m = re.search(rf'<Rate currency="{symbol}">([\d.]+)</Rate>', body)
                if m:
                    rows.append((d, float(m.group(1))))
        s = pd.Series(dict(rows))
    else:
        raise ValueError(f'unknown source: {source}')
    s = pd.to_numeric(s, errors='coerce')
    s.index = pd.to_datetime(s.index)
    s.index.name = 'Date'
    return s.sort_index().loc[start:end].dropna()


def load(name, start=START, end=END):
    """Seria cu numele din SERIES (pentru aur si EUR/USD: doar zilele lucratoare)."""
    source, symbol, field = SERIES[name]
    s = fetch_daily(symbol, source, field, start, end).rename(name)
    if name in WEEKDAYS_ONLY:
        s = s[s.index.dayofweek < 5]
    return s


def load_panel(names, start=START, end=END):
    """Mai multe serii aliniate pe data (NaN acolo unde o piata nu tranzactioneaza)."""
    return pd.concat([load(n, start, end) for n in names], axis=1)


## Helper functions

In [ ]:
CROSS = ['S&P 500 TR (SPY)', 'Euro Stoxx 50 (USD)', 'Nikkei 225 (USD)', 'Gold', 'US Treasuries 20y+ (TLT)', 'Bitcoin']
CROSS_COL = [MainBlue, Forest, Purple, Amber, '#17A2B8', Orange]
OFFSETS = {'Gold': (-7, 2, 'right'), 'S&P 500 TR (SPY)': (0, 8, 'center'), 'Nikkei 225 (USD)': (7, 2, 'left'), 'Euro Stoxx 50 (USD)': (7, -7, 'left'), 'US Treasuries 20y+ (TLT)': (-7, -9, 'right'), 'EUR/USD': (0, 7, 'center'), 'EUR/RON': (6, 3, 'left'), 'Bitcoin': (-6, 3, 'right'), 'Ethereum': (-6, 3, 'right')}

def ann_stats(p, periods=252):
    """Randament anualizat (CAGR) si volatilitate anualizata din preturi zilnice.

    periods='obs' foloseste numarul efectiv de observatii pe an al seriei.
    """
    p = p.dropna()
    years = (p.index[-1] - p.index[0]).days / 365.25
    cagr = (p.iloc[-1] / p.iloc[0]) ** (1 / years) - 1
    r = np.log(p).diff().dropna()
    if periods == 'obs':
        periods = len(r) / years
    return cagr, r.std() * np.sqrt(periods)


def to_usd(local, fx, invert=False):
    """Converteste un indice in moneda locala in USD cu cursul FRED din aceeasi zi
    (ultimul curs disponibil, cel mult 5 zile in urma, daca ziua lipseste)."""
    f = fx.reindex(local.index, method='ffill', limit=5)
    return (local / f if invert else local * f).dropna()


def cross_panel():
    """Seriile comparate pe clase de active, toate in USD:
    S&P 500 = SPY ajustat (randament total); Euro Stoxx 50 si Nikkei 225 = indici de pret convertiti in USD;
    TLT = randament total; aur = spot; Bitcoin = pret."""
    return {
        'S&P 500 TR (SPY)': px['SPY'].dropna(),
        'Euro Stoxx 50 (USD)': to_usd(px['Euro Stoxx 50'].dropna(), fx_usd['USD per EUR']),
        'Nikkei 225 (USD)': to_usd(px['Nikkei 225'].dropna(), fx_usd['JPY per USD'], invert=True),
        'Gold': px['Gold'].dropna(),
        'US Treasuries 20y+ (TLT)': px['US Treasuries 20y+ (TLT)'].dropna(),
        'Bitcoin': px['Bitcoin'].dropna(),
        'Ethereum': px['Ethereum'].dropna(),
        'EUR/USD': px['EUR/USD'].dropna(),
    }


def dividend_gap(start='2015-01-02'):
    """Diferenta de CAGR dintre S&P 500 cu dividende reinvestite (SPY ajustat) si indicele de pret."""
    c_tr, _ = ann_stats(px['SPY'].loc[start:])
    c_px, _ = ann_stats(px['S&P 500'].loc[start:])
    return c_tr, c_px, c_tr - c_px

## Data

In [ ]:
px = load_panel(['S&P 500', 'SPY', 'Euro Stoxx 50', 'Nikkei 225', 'Gold', 'US Treasuries 20y+ (TLT)', 'Bitcoin', 'Ethereum', 'EUR/USD'])
fx_usd = load_panel(['USD per EUR', 'JPY per USD'])
eurron = load('EUR/RON')

## Charts

In [ ]:
def fig_cross_asset_growth(start='2015-01-02'):
    fig, ax = plt.subplots(figsize=(7.0, 3.2))
    out = {}
    panel = cross_panel()
    for name, c in zip(CROSS, CROSS_COL):
        s = panel[name].loc[start:].dropna()
        g = s / s.iloc[0]
        ax.plot(g.index, g.values, color=c, lw=0.9 if name != 'Bitcoin' else 0.8,
                label=f'{name} (x{g.iloc[-1]:.1f})')
        out[name] = g.iloc[-1]
    log_axis(ax, [0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500])
    ax.axhline(1, color=Gray, lw=0.4, ls=':')
    ax.set_ylabel('Growth of 1 USD (log scale)')
    ax.set_title(f'{pd.Timestamp(start).year}-2026, all in USD: S&P 500 and TLT = total return, '
                 'Euro Stoxx 50 and Nikkei 225 = price, gold = spot',
                 fontsize=8.5, loc='left')
    legend_outside_bottom(ax, ncol=3, y=-0.12)
    plt.tight_layout()
    save_fig('ch0_cross_asset_growth')
    return pd.Series(out)

In [ ]:
def fig_risk_return(start='2015-01-02'):
    names = CROSS + ['Ethereum', 'EUR/USD', 'EUR/RON']
    cols = CROSS_COL + [Purple, Crimson, IDAred]
    rows = []
    panel = cross_panel()
    for n in names:
        s = (eurron if n == 'EUR/RON' else panel[n]).loc[start:].dropna()
        periods = 365 if n in ('Bitcoin', 'Ethereum') else ('obs' if n in ('Gold', 'EUR/USD') else 252)
        cagr, v = ann_stats(s, periods)
        rows.append((n, cagr, v, s.index[0]))
    res = pd.DataFrame(rows, columns=['asset', 'cagr', 'vol', 'from']).set_index('asset')
    fig, ax = plt.subplots(figsize=(6.6, 3.2))
    for (n, r), c in zip(res.iterrows(), cols):
        ax.scatter(r['vol'], r['cagr'], s=40, color=c, zorder=3)
        late = r['from'] > pd.Timestamp(start) + pd.Timedelta(days=30)
        lab = f"{n} (from {r['from']:%b %Y})" if late else n
        dx, dy, ha = OFFSETS.get(n, (5, 3, 'left'))
        ax.annotate(lab, (r['vol'], r['cagr']), xytext=(dx, dy), textcoords='offset points',
                    fontsize=7.5, ha=ha)
    ax.set_xscale('log')
    ax.set_xticks([0.02, 0.05, 0.1, 0.2, 0.4, 0.8])
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.axhline(0, color=Gray, lw=0.5)
    ax.set_ylim(-0.08, None)
    ax.set_xlabel('Annualised volatility (log scale)')
    ax.set_ylabel('Annualised return (CAGR)')
    ax.set_title('Risk and return in USD, 2015-2026: higher volatility is not always rewarded',
                 fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch0_risk_return')
    return res

## Run

In [ ]:
print(fig_cross_asset_growth().round(2))
print(fig_risk_return().round(3))
print('S&P 500 CAGR with dividends, price only, gap:', [round(x, 4) for x in dividend_gap()])

![ch0_cross_asset_growth](./ch0_cross_asset_growth.png)

![ch0_risk_return](./ch0_risk_return.png)

Output: `ch0_cross_asset_growth.pdf`, `ch0_risk_return.pdf`